# Qwen2-VL-2B QLoRA 微调：机械臂关节角度预测

基于本地生成的多任务专家轨迹数据集（`qwen_train.json`），用 4-bit QLoRA 微调
`Qwen/Qwen2-VL-2B-Instruct`，使其根据相机画面 + 中文指令输出下一步的 7 个关节角度。

流程：安装依赖 → 挂载 Drive 并加载数据 → 构造 Dataset → 4-bit 加载模型 → 注入 LoRA → Trainer 训练 → 保存权重。

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets qwen-vl-utils

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import os
import re

DATA_DIR = "/content/drive/MyDrive/vla_dataset/expert_multi_v1"

with open(os.path.join(DATA_DIR, "qwen_train.json"), encoding="utf-8") as f:
    records = json.load(f)
print(f"已加载 {len(records)} 帧训练样本")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

# --- 按 episode 划分 train/val ---
def _episode_id(rec):
    m = re.search(r"ep_(\d+)_step_", rec["image"])
    return int(m.group(1)) if m else -1

with open(os.path.join(DATA_DIR, "episode_split.json"), encoding="utf-8") as f:
    split = json.load(f)

train_ids = set(split["train"])
val_ids   = set(split["val"])

train_records = [r for r in records if _episode_id(r) in train_ids]
val_records   = [r for r in records if _episode_id(r) in val_ids]
print(f"train: {len(train_records)} 帧, val: {len(val_records)} 帧")

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 1280 * 28 * 28


def _strip_image_placeholder(text):
    return re.sub(r"<image>\s*", "", text)


class QwenVLADataset(Dataset):

    def __init__(self, records, processor, image_base_dir):
        self.records = records
        self.processor = processor
        self.image_base_dir = image_base_dir

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image_path = os.path.join(self.image_base_dir, os.path.basename(rec["image"]))
        image = Image.open(image_path).convert("RGB")

        human_value = rec["conversations"][0]["value"]
        gpt_value   = rec["conversations"][1]["value"]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": _strip_image_placeholder(human_value)},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": gpt_value}],
            },
        ]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = self.processor(text=[text], images=[image], return_tensors="pt")

        input_ids = inputs["input_ids"][0]
        attention_mask = inputs["attention_mask"][0]
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        assistant_token = self.processor.tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)
        assistant_start = None
        for i in range(len(input_ids) - len(assistant_token) + 1):
            if input_ids[i : i + len(assistant_token)].tolist() == assistant_token:
                assistant_start = i
                break
        if assistant_start is not None:
            labels[:assistant_start] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "pixel_values": inputs["pixel_values"],
            "image_grid_thw": inputs["image_grid_thw"],
            "mm_token_type_ids": inputs["mm_token_type_ids"][0],
        }


train_dataset = QwenVLADataset(train_records, processor, DATA_DIR)
val_dataset   = QwenVLADataset(val_records,   processor, DATA_DIR)
print(f"train dataset: {len(train_dataset)}, val dataset: {len(val_dataset)}")

sample = train_dataset[0]
print(f"input_ids: {sample['input_ids'].shape}")
print(f"pixel_values: {sample['pixel_values'].shape}")
print(f"image_grid_thw: {sample['image_grid_thw'].shape}")
print(f"mm_token_type_ids: {sample['mm_token_type_ids'].shape}")

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print(f"已加载 Qwen2-VL-2B-Instruct 4bit，参数量: {model.num_parameters():,}")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="qwen2_vl_vla_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)


def collate_fn(batch):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
    pad_token_id = processor.tokenizer.pad_token_id

    input_ids = torch.nn.utils.rnn.pad_sequence(
        [b["input_ids"] for b in batch], batch_first=True, padding_value=pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch], batch_first=True, padding_value=-100)
    mm_token_type_ids = torch.nn.utils.rnn.pad_sequence(
        [b["mm_token_type_ids"] for b in batch], batch_first=True, padding_value=0)

    pvs = [b["pixel_values"].squeeze(0) for b in batch]
    gts = [b["image_grid_thw"].squeeze(0) for b in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "pixel_values": torch.cat(pvs, dim=0),
        "image_grid_thw": torch.stack(gts, dim=0),
        "mm_token_type_ids": mm_token_type_ids,
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    processing_class=processor,
)

In [ ]:
trainer.train()

In [ ]:
final_dir = "/content/drive/MyDrive/vla_project/qwen2_vl_vla_lora_final"
model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
print(f"LoRA 权重已保存到: {final_dir}")

LOCAL_USE = """\
本地使用说明:
1) 安装依赖: pip install transformers peft bitsandbytes accelerate qwen-vl-utils
2) 加载 4bit 基础模型 + LoRA adapter:
     from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
     from peft import PeftModel
     base = Qwen2VLForConditionalGeneration.from_pretrained(
         "Qwen/Qwen2-VL-2B-Instruct", device_map="auto"
     )
     model = PeftModel.from_pretrained(base, final_dir)
3) （可选）合并权重以加速推理:
     model = model.merge_and_unload()
     model.save_pretrained("qwen2_vl_vla_merged")
     processor.save_pretrained("qwen2_vl_vla_merged")
4) 推理: 用 processor.apply_chat_template 构造 messages
   （user 含 {\"type\": \"image\"} + 中文指令），注意 add_generation_prompt=True，
   generate 输出的文本即 7 个关节角度（逗号分隔，4 位小数），解析后喂给仿真环境。
"""
print(LOCAL_USE)